---
---

# **Employee Burnout Risk: Inference**

*Authored by Sean Kafka Adhyaksa*

---
---

This notebook demonstrates how the saved multimodal model can be used to predict burnout risk for a new employee record. The same saved preprocessing objects from training are reused here so the inference flow stays consistent with the modeling pipeline.

# **Importing Libraries**

In [11]:
import ast
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model


---

# **Loading Model**

In [12]:
# Load saved preprocessing artifacts and the trained multimodal model.
model_dir = Path('model_saving')

with open(model_dir / 'list_num_cols.txt', 'r') as file_1:
    list_num_cols = json.load(file_1)

with open(model_dir / 'list_cat_cols.txt', 'r') as file_2:
    list_cat_cols = json.load(file_2)

with open(model_dir / 'list_feedback_col.txt', 'r') as file_3:
    list_feedback_col = json.load(file_3)

with open(model_dir / 'list_skills_tokens.txt', 'r') as file_4:
    list_skill_tokens = json.load(file_4)

with open(model_dir / 'scaler.pkl', 'rb') as file_5:
    scaler = pickle.load(file_5)

with open(model_dir / 'encoder.pkl', 'rb') as file_6:
    encoder = pickle.load(file_6)

model_base = load_model(model_dir / 'model_base.keras')


---

# **Model Inferencing**

## **1. Create dummy data**

In [13]:
# Create one sample employee record for inference.
data_inf = pd.DataFrame({
    'job_level': ['Entry'],
    'left_company': ['No'],
    'department_group': ['HR'],
    'tenure_months': [9],
    'salary': [69575.36],
    'performance_score': [0.68],
    'satisfaction_score': [0.43],
    'workload_score': [0.83],
    'team_sentiment': [0.41],
    'project_completion_rate': [0.72],
    'overtime_hours': [11],
    'training_participation': [0.58],
    'collaboration_score': [0.47],
    'email_sentiment': [0.39],
    'role_complexity_score': [0.20],
    'career_progression_score': [0.27],
    'feedback_tokens': ['heavy workload long hours unclear direction limited support difficult maintain work life balance'],
    'skillset': [[
        'Time Management',
        'Critical Thinking',
        'Communication',
        'Teamwork',
        'Adaptability',
        'Python',
        'SQL',
        'Pandas'
    ]]
})

data_inf


,job_level,left_company,department_group,tenure_months,salary,performance_score,satisfaction_score,workload_score,team_sentiment,project_completion_rate,overtime_hours,training_participation,collaboration_score,email_sentiment,role_complexity_score,career_progression_score,feedback_tokens,skillset
0,Entry,No,HR,9,69575.36,0.68,0.43,0.83,0.41,0.72,11,0.58,0.47,0.39,0.2,0.27,heavy workload long hours unclear direction li...,"[Time Management, Critical Thinking, Communica..."


## **2. Filtering Features for Model Input**

In [14]:
def parse_skillset_text(value):
    """Convert a list-like skillset value into plain text for the text branch."""
    if isinstance(value, list):
        return ' '.join(str(item) for item in value)

    if pd.isna(value):
        return ''

    text = str(value)

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return ' '.join(str(item) for item in parsed)
    except (ValueError, SyntaxError):
        pass

    return text


In [15]:
# Prepare tabular inputs with the same encoder and scaler used during training.
data_inf_cat = data_inf[list_cat_cols]
data_inf_num = data_inf[list_num_cols]

data_inf_cat_encoded = encoder.transform(data_inf_cat)
data_inf_num_scaled = scaler.transform(data_inf_num)
data_inf_tab = np.concatenate([data_inf_cat_encoded, data_inf_num_scaled], axis=1)

# Prepare text inputs for both language branches.
feedback_col = list_feedback_col[0]
skill_col = list_skill_tokens[0]

data_inf_feedback = data_inf[feedback_col].fillna('').astype(str).values
data_inf_skillset = data_inf[skill_col].apply(parse_skillset_text).values

## **3. Predict!**

In [16]:
def classify_burnout(score):
    """Map the predicted burnout risk into a simple descriptive label."""
    if score < 0.30:
        return 'Low Burnout'
    if score < 0.60:
        return 'Medium Burnout'
    return 'High Burnout'


prediction = model_base.predict(
    {
        'tabular_input': data_inf_tab,
        'feedback_input': data_inf_feedback,
        'skills_input': data_inf_skillset
    },
    verbose=0
)

predicted_burnout_risk = float(prediction.ravel()[0])
predicted_burnout_level = classify_burnout(predicted_burnout_risk)

result = pd.DataFrame({
    'predicted_burnout_risk': [round(predicted_burnout_risk, 4)],
    'predicted_burnout_level': [predicted_burnout_level]
})

result


,predicted_burnout_risk,predicted_burnout_level
0,0.8675,High Burnout
